# Optional Part 10: Tiny Transformer for AI Engineers — Revision Audit

This is an optional systems-oriented revision, not a second Transformer course. It reads the canonical Parts 6-8 artifacts and asks six engineering questions:

1. Is the data contract explicit and hash-verified?
2. Does the model contract agree with tokenizer and shard IDs?
3. Is training evidence measured rather than narrated?
4. Was selection based on held-out validation?
5. Can the selected package reload and resume?
6. What does the tiny run still not prove?

Run Parts 6, 7, and 8B first. This notebook is read-only with respect to their artifacts.


In [ ]:
# Imports and fail-closed artifact discovery
import hashlib
import json
from pathlib import Path
import numpy as np
import torch

ROOT = Path("artifacts/base-lm")
required = [
    "model-config.json",
    "dataset-manifest.json",
    "training-manifest.json",
    "train.bin",
    "validation.bin",
    "final/model.pt",
    "final/model-config.json",
    "final/checkpoint-manifest.json",
]
missing = [name for name in required if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError(
        "Run Parts 6, 7, and 8B first. Missing: " + ", ".join(missing)
    )
print("required artifacts found:", len(required))


## 1. Data contract: bytes, dtype, and block shape

A `Dataset`/`DataLoader` with dynamic collation is ideal for natural-length sentences in Part 4. Pretraining owns a different contract: fixed-length packed token streams, memory-mapped shards, and manifests. Do not collapse those two workloads into one abstraction.


In [ ]:
# Verify shard identities and inspect fixed-length packed blocks
dataset_manifest = json.loads((ROOT / "dataset-manifest.json").read_text())
training_manifest = json.loads((ROOT / "training-manifest.json").read_text())
upstream_config = json.loads((ROOT / "model-config.json").read_text())
config = json.loads((ROOT / "final/model-config.json").read_text())

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

train_hash = sha256(ROOT / "train.bin")
validation_hash = sha256(ROOT / "validation.bin")
assert train_hash == training_manifest["train_sha256"]
assert validation_hash == training_manifest["validation_sha256"]

dtype = np.dtype(dataset_manifest["dtype"])
train_tokens = np.memmap(ROOT / "train.bin", mode="r", dtype=dtype)
context_length = config["context_length"]
usable = (len(train_tokens) // (context_length + 1)) * (context_length + 1)
blocks = train_tokens[:usable].reshape(-1, context_length + 1)
print("train token count:", len(train_tokens))
print("packed block shape:", blocks.shape)
print("first input/target shapes:", blocks[0, :-1].shape, blocks[0, 1:].shape)
print("Part 6 seed vocabulary:", upstream_config["vocab_size"])
print("tokenizer-resolved vocabulary:", config["vocab_size"])


## 2. Model contract: inspect the package rather than trusting a class name

The config defines architecture and dimensions. The checkpoint contains learned tensors. Their keys and shapes provide a second, independently checkable view of what was actually saved.


In [ ]:
# Inspect saved tensor surfaces without rebuilding the whole lesson
model_state = torch.load(
    ROOT / "final/model.pt", map_location="cpu", weights_only=True
)
tensor_count = sum(isinstance(value, torch.Tensor) for value in model_state.values())
parameter_count = sum(
    value.numel() for value in model_state.values() if isinstance(value, torch.Tensor)
)
maximum_token_id = int(train_tokens.max())
resolved_vocab_size = int(dataset_manifest["vocab_size"])
print("architecture:", config["architecture"])
print("saved tensors:", tensor_count)
print("saved scalar parameters:", parameter_count)
print("maximum train token ID:", maximum_token_id)
print("architecture-seed vocabulary:", config["vocab_size"])
print("manifest-resolved vocabulary:", resolved_vocab_size)
print("sample keys:")
for key in list(model_state)[:8]:
    value = model_state[key]
    print(f"  {key}: {tuple(value.shape)}")
assert maximum_token_id < resolved_vocab_size
assert model_state["token_embedding.weight"].shape[0] == resolved_vocab_size


## 3. Training evidence and selection contract

The latest checkpoint is not automatically the best. The manifest must name the selected checkpoint, the held-out metric, and the exact resume-parity result.


In [ ]:
# Audit measured training, validation selection, and resume parity
selected = Path(
    training_manifest["selected_checkpoint"].replace("\\", "/")
)
selected_path = selected if selected.is_absolute() else Path.cwd() / selected
parity = training_manifest["resume_parity"]
print("optimizer steps:", training_manifest["optimizer_settings"]["optimizer_steps"])
print("selection metric:", training_manifest["selection_metric"])
print("selected checkpoint:", selected_path)
print("final validation loss:", training_manifest["final_validation_loss"])
print("resume loss delta:", parity["loss_delta"])
print("resume parameter delta:", parity["max_parameter_delta"])
assert selected_path.exists()
assert parity["loss_delta"] <= parity["tolerance"]
assert parity["max_parameter_delta"] <= parity["tolerance"]


## 4. Package contract: inference state versus continuation state

`final/model.pt` is enough for model reconstruction only when paired with config and tokenizer. Faithful training continuation additionally needs optimizer, scheduler, counters, and RNG state.


In [ ]:
# Verify final package surfaces and lineage digests
checkpoint_manifest = json.loads(
    (ROOT / "final/checkpoint-manifest.json").read_text()
)
final_required = [
    "model.pt",
    "optimizer.pt",
    "scheduler.pt",
    "counters.json",
    "rng.pt",
    "model-config.json",
    "training-manifest.json",
]
absent = [name for name in final_required if not (ROOT / "final" / name).exists()]
if absent:
    raise FileNotFoundError("Incomplete final package: " + ", ".join(absent))
print("final package files verified:", len(final_required))
print("checkpoint schema:", checkpoint_manifest["schema_version"])
print("config digest:", training_manifest["config_digest"][:16] + "...")
print("dataset digest:", training_manifest["dataset_manifest_digest"][:16] + "...")
print("tokenizer digest:", training_manifest["tokenizer_digest"][:16] + "...")


## Revision scorecard

| Contract | Audit result |
|---|---|
| Data | shard hashes matched; packed block shapes were reconstructed |
| Model | checkpoint tensors and token-ID bounds agreed with config |
| Training | optimizer-step and validation evidence came from the manifest |
| Evaluation | selection metric was held-out token-weighted validation loss |
| Artifacts | inference and continuation surfaces were both present |
| Resume | recorded loss and parameter deltas met tolerance |

### What this tiny run does not prove

- broad knowledge or reasoning;
- instruction following or assistant behavior;
- factuality, safety, or production readiness;
- scalability of this single-process CPU profile;
- quality outside the local corpus.

The durable engineering habit is: **trace every claim back to a tensor shape, a measured metric, or a hashed artifact.**
